# Chat with a small language model in your browser

This notebook runs a real language model **entirely on your own computer**, inside the browser. Nothing is sent to a server.

**Before you start:** use Chrome or Edge on a laptop, and be on good wifi. Models download once and are then saved (cached) in your browser, so you never have to download the same model twice.

**The plan:** see what models exist, download one (or more), check what you have saved, then load one and chat with it.

## Step 1: Get set up

This checks that your browser can run models (using WebGPU, your computer's graphics chip), loads the WebLLM library, and sets up a few helpers the later steps use. You do not need to read the code in this cell.

In [ ]:
import js
from pyodide.ffi import to_js, create_proxy
from IPython.display import display

if getattr(js.navigator, "gpu", None):
    print("WebGPU is available. You can run models.")
else:
    print("WebGPU is not available here. Try the latest Chrome or Edge on a laptop.")

webllm = await js.eval("import('https://esm.run/@mlc-ai/web-llm')")
print("WebLLM loaded.")

engine = None        # the thing that runs a model
loaded_model = None  # which model is currently loaded


def js_obj(d):
    """Turn a Python dict into a JavaScript object."""
    return to_js(d, dict_converter=js.Object.fromEntries)


def progress_bar():
    """Show a progress bar that updates in place."""
    status = display("Starting...", display_id=True)

    def show(report):
        filled = int(report.progress * 30)
        bar = "#" * filled + "-" * (30 - filled)
        status.update(f"[{bar}] {report.progress:.0%}\n{report.text}")

    return status, create_proxy(show)


async def start_model(model_id):
    """Get a model ready to run. Downloads it only if it is not already saved."""
    global engine
    status, progress = progress_bar()

    async def attempt(chat_options=None):
        global engine
        if engine is None:
            if chat_options:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}), js_obj(chat_options))
            else:
                engine = await webllm.CreateMLCEngine(
                    model_id, js_obj({"initProgressCallback": progress}))
        else:
            engine.setInitProgressCallback(progress)
            if chat_options:
                await engine.reload(model_id, js_obj(chat_options))
            else:
                await engine.reload(model_id)

    try:
        await attempt()
    except Exception as error:
        # Some models have two conflicting memory-window settings.
        # If so, turn off the "sliding window" one and try again.
        if "WindowSizeConfigurationError" in str(error):
            status.update("Adjusting this model's settings and retrying...")
            await attempt({"sliding_window_size": -1})
        else:
            raise

## Step 2: See which models are available

WebLLM comes with a long list of ready-to-use models. You can narrow it down with three settings at the top of the next cell:

- **`NEWER_THAN`**: only show models that came out on or after this month (`"YYYY-MM"`). Set it to `""` to show all dates.
- **`MAX_SIZE_MB`**: only show models that need less than this much graphics memory. 4000 MB is fine on most newer laptops; use 2000 for older or lower-powered ones.
- **`MODEL_TYPE`**: what kind of model to show.
  - `"chat"` models answer questions. This is what we use in this notebook.
  - `"embedding"` models turn text into lists of numbers, used for search. We will use these later.
  - `"vision"` models can look at images.

Each model also comes in a few versions. We only show the **`q4f16`** version, which is compressed to about a quarter of the original size and is the best choice for laptops.

**About release dates:** WebLLM does not record when models came out, so the cell below includes a small table, `RELEASED`, with the month each model family was first released. These dates are approximate. If a new family shows up with a `?`, you can add it to the table.

In [ ]:
NEWER_THAN = "2025-01"
MAX_SIZE_MB = 4000
MODEL_TYPE = "chat"

# When each model family first came out (approximate, "YYYY-MM").
# More specific names must come before less specific ones (Qwen3.5 before Qwen3).
RELEASED = [
    ("Qwen3.5-", "2026-03"),
    ("Ministral-3-", "2025-12"),
    ("Qwen3-", "2025-04"),
    ("OLMo-2-0425", "2025-04"),
    ("gemma3-", "2025-03"),
    ("Phi-4-mini", "2025-02"),
    ("DeepSeek-R1-Distill", "2025-01"),
    ("SmolLM2-", "2024-11"),
    ("OLMo-2-1124", "2024-11"),
    ("Qwen2.5-Coder", "2024-11"),
    ("Qwen2.5-", "2024-09"),
    ("Llama-3.2-", "2024-09"),
    ("Phi-3.5", "2024-08"),
    ("Hermes-3", "2024-08"),
    ("gemma-2-", "2024-07"),
    ("Llama-3.1-", "2024-07"),
    ("Qwen2-", "2024-06"),
    ("Mistral-7B-Instruct-v0.3", "2024-05"),
    ("Llama-3-", "2024-04"),
    ("Phi-3-", "2024-04"),
    ("snowflake-arctic-embed", "2024-04"),
    ("Hermes-2-Theta", "2024-05"),
    ("Hermes-2-Pro", "2024-03"),
    ("gemma-2b-it", "2024-02"),
    ("TinyLlama", "2024-01"),
    ("stablelm-2-zephyr", "2024-01"),
    ("phi-2", "2023-12"),
    ("Mistral-7B-Instruct-v0.2", "2023-12"),
    ("NeuralHermes-2.5", "2023-12"),
    ("WizardMath-7B-V1.1", "2023-12"),
    ("OpenHermes-2.5", "2023-11"),
    ("phi-1_5", "2023-09"),
    ("Llama-2", "2023-07"),
    ("RedPajama", "2023-05"),
]

TYPES = {0: "chat", 1: "embedding", 2: "vision"}


def released(model_id):
    for prefix, month in RELEASED:
        if model_id.lower().startswith(prefix.lower()):
            return month
    return "?"


# Gather every model WebLLM knows about
all_models = []
for m in webllm.prebuiltAppConfig.model_list:
    size = getattr(m, "vram_required_MB", None)
    all_models.append({
        "name": m.model_id,
        "size": int(size) if size else None,
        "type": TYPES.get(getattr(m, "model_type", None) or 0, "other"),
        "released": released(m.model_id),
    })

# Apply the filters
models = []
for m in all_models:
    if m["type"] != MODEL_TYPE:
        continue
    if "q4f16" not in m["name"]:
        continue
    if not m["size"] or m["size"] > MAX_SIZE_MB:
        continue
    if NEWER_THAN and (m["released"] == "?" or m["released"] < NEWER_THAN):
        continue
    models.append(m)

models.sort(key=lambda m: (m["released"], m["size"]), reverse=True)

print(f"{len(models)} models match your filters (newest first)\n")
print(f"{'#':>3}  {'released':<9} {'size (MB)':>9}  {'saved?':<7} model")
for number, m in enumerate(models, start=1):
    saved = "saved" if await webllm.hasModelInCache(m["name"]) else ""
    print(f"{number:>3}  {m['released']:<9} {m['size']:>9}  {saved:<7} {m['name']}")

### Optional: browse the full model list

This shows **every** model WebLLM supports as a table, with no filters except an optional search word. Put a word in `SEARCH` (like `"qwen"` or `"phi"`) to narrow it down, or leave it empty to see everything.

In [ ]:
import pandas as pd

SEARCH = ""

table = pd.DataFrame(all_models).rename(columns={"size": "size (MB)"})
table = table.sort_values(["released", "size (MB)"], ascending=[False, True]).reset_index(drop=True)
if SEARCH:
    table = table[table["name"].str.contains(SEARCH, case=False)]

pd.set_option("display.max_rows", None)
print(f"{len(table)} models")
table

## Step 3: Pick a model to download

Type the **number** of a model from the list above. A small one near the top is a good place to start.

In [ ]:
DOWNLOAD_CHOICE = 1

download_name = models[DOWNLOAD_CHOICE - 1]["name"]
download_size = models[DOWNLOAD_CHOICE - 1]["size"]
print(f"You picked: {download_name} (about {download_size} MB)")

## Step 4: Download it

This saves the model in your browser. If it is already saved, nothing is downloaded. You can repeat Steps 3 and 4 to download more than one model.

In [ ]:
if await webllm.hasModelInCache(download_name):
    print(f"{download_name} is already saved. Nothing to download.")
else:
    await start_model(download_name)
    await engine.unload()   # downloaded and saved; free up memory for now
    loaded_model = None
    print(f"{download_name} is downloaded and saved in your browser.")

## Step 5: Check what you have saved

This lists every model saved in your browser, numbered so you can pick one to load in Step 6.

In [ ]:
saved_models = []
for m in all_models:
    if await webllm.hasModelInCache(m["name"]):
        saved_models.append((m["name"], m["size"]))

if saved_models:
    print("Models saved in your browser:")
    for number, (name, size) in enumerate(saved_models, start=1):
        print(f"{number:>3}  {name}  ({size or '?'} MB)")
    total = sum(size or 0 for _, size in saved_models)
    print(f"\nTotal: about {total} MB")
else:
    print("No models saved yet. Go back to Steps 3 and 4.")

**Optional: free up space.** Put the number of a saved model (from the list just above) in `DELETE_CHOICE` and run the cell to remove it. Leave it at 0 to do nothing.

In [ ]:
DELETE_CHOICE = 0

if DELETE_CHOICE:
    name, _ = saved_models[DELETE_CHOICE - 1]
    await webllm.deleteModelAllInfoInCache(name)
    print(f"Deleted {name}. Run Step 5 again to see the updated list.")
else:
    print("Nothing deleted.")

## Step 6: Load a model

Pick the **number** of a saved model from Step 5. Loading takes a few seconds while the model moves onto your graphics chip. No downloading happens here.

To switch models later, change `LOAD_CHOICE` and run this cell again.

In [ ]:
LOAD_CHOICE = 1

if not saved_models:
    print("No saved models yet. Go back to Steps 3 and 4.")
else:
    model_name, _ = saved_models[LOAD_CHOICE - 1]
    await start_model(model_name)
    loaded_model = model_name
    print(f"{loaded_model} is loaded and ready!")

## Step 7: Talk to the model, one piece at a time

Chatting with a model takes three moves: **write a message**, **send it**, and **read the reply**. We will do each one separately so you can see what is going on, then bundle them into a handy shortcut.

### 7a: Write a message

A chat model reads a **list of messages**. Each message is a Python dictionary with two parts:

- `"role"`: who is speaking. `"user"` is you.
- `"content"`: what they said.

Right now our conversation is just one message from you.

In [ ]:
messages = [
    {"role": "user", "content": "Write a poem about bike riding in Amsterdam."}
]

messages

### 7b: Choose settings and package the request

Along with the messages, we can send settings. `temperature` controls how random the answer is (more on this in Step 8).

The model runs in JavaScript, not Python, so `js_obj()` translates our Python dictionary into a form JavaScript understands.

In [ ]:
request = js_obj({
    "messages": messages,
    "temperature": 0.7,
})

print("Request is ready to send.")

### 7c: Send it and wait for the reply

`await` means "wait here until the model finishes." The model writes its answer one small piece at a time, so this can take a few seconds.

In [ ]:
if loaded_model is None:
    print("No model is loaded yet. Run Step 6 until it says the model is ready.")
else:
    reply = await engine.chat.completions.create(request)
    print("Got a reply!")

### 7d: Look inside the reply

The reply contains more than just the answer. Let's see the whole thing first.

In [ ]:
from pprint import pprint

pprint(reply.to_py())

That is a lot! The parts worth knowing:

- **`choices[0].message.content`**: the model's actual answer.
- **`choices[0].finish_reason`**: why it stopped. `"stop"` means it finished on its own; `"length"` means it ran out of room.
- **`usage`**: how many **tokens** (small chunks of words) went in and came out. Models read and write in tokens, not whole words.

In [ ]:
answer = reply.choices[0].message.content
print(answer)

In [ ]:
print("Why it stopped:   ", reply.choices[0].finish_reason)
print("Tokens you sent:  ", reply.usage.prompt_tokens)
print("Tokens it wrote:  ", reply.usage.completion_tokens)

### 7e: Bundle it into a shortcut

Doing all of that every time would be tedious. Here we put steps 7a to 7c into one function called `ask()`. Give it a question, and it gives back the answer.

In [ ]:
async def ask(question, temperature=0.7):
    if loaded_model is None:
        return "No model is loaded yet. Run Step 6 until it says the model is ready."
    messages = [{"role": "user", "content": question}]                # 7a: write
    request = js_obj({"messages": messages, "temperature": temperature})  # 7b: package
    reply = await engine.chat.completions.create(request)               # 7c: send
    return reply.choices[0].message.content                             # 7d: read

print(await ask("Write a poem about bike riding in Amsterdam."))

## Step 8: Experiment with temperature

Temperature controls randomness. Low temperature gives similar answers every time; high temperature gives more varied (and sometimes stranger) answers. Run this cell a few times.

In [ ]:
question = "Write a one-line slogan for a coffee shop."

for t in [0.0, 0.7, 1.5]:
    answer = await ask(question, temperature=t)
    print(f"temperature={t}: {answer}\n")

## Your turn

Try your own questions below. Where is this model good? Where does it make mistakes? Load a different model in Step 6, ask the same question, and compare.

In [ ]:
print(await ask("What is the capital of Australia?"))